In [9]:
import pybamm
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

# import  settings
import plot_settings
plot_settings.apply()

In [10]:
# load model
model = pybamm.lithium_ion.SPM()

# create geometry
geometry = model.default_geometry

# load parameter values and process model and geometry
param = model.default_parameter_values

def my_current(t):
    return 1

param["Current function [A]"] = 1 

param.process_geometry(geometry)
param.process_model(model)

# create simulation
simulation = pybamm.Simulation(model, parameter_values=param)

# set mesh
var = pybamm.standard_spatial_vars
var_pts = {var.x_n: 30, var.x_s: 30, var.x_p: 30, var.r_n: 10, var.r_p: 10}
mesh = pybamm.Mesh(geometry, model.default_submesh_types, var_pts)

# discretise model
disc = pybamm.Discretisation(mesh, model.default_spatial_methods)
disc.process_model(model);

In [11]:
# solve model
t_eval = [0, 3600]
solver = pybamm.IDAKLUSolver(atol=1e-6, rtol=1e-3)
solution = solver.solve(model, t_eval)

In [12]:
c_n = solution["Negative particle concentration"].entries
r_n = solution["r_n [m]"].entries
x_n = solution["x_n [m]"].entries
t = solution["Time [s]"].entries

print(c_n.shape)  # (nr, nx, nt)
print(r_n.shape)  # (nr, nx, nt)
print(x_n.shape)  # (nx, nt)
print(t.shape)    # (nt,)

(10, 30, 35)
(10, 30, 35)
(30, 35)
(35,)


In [13]:
# # Split data into training, validation, and test sets
# X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
# X_valid, X_test, y_valid, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
# # I = 1
# # t_tot = 3600
# c0 = solution["Positive particle concentration [mol.m-3]"].entries[:,0,0]
# print(c0)
# data_shape = 2 + len(c0)


input_shape = 3
output_shape = (100, 20)
print((input_shape,))

(3,)


In [ ]:
model = keras.models.Sequential([   
    keras.layers.Input(shape=(data_shape,)),
    keras.layers.Dense(64, activation='tanh'),
    keras.layers.Dense(32, activation='tanh'),
    keras.layers.Dense(output_shape, activation='relu')
])

In [ ]:
model.compile(
  optimizer='adam',
  loss='mse',metrics=['accuracy'])

In [ ]:
epochs = 150     ## Number of epoch to run
batch_size = 32  ## Mini batch size

es = keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights = True)
# Train the model
history = model.fit(
  X_train, y_train,
  epochs=epochs,
  batch_size=batch_size,
  verbose = 1,
  callbacks=[es],
  validation_data=(X_valid, y_valid))

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

plt.plot(train_loss, label='Training loss')
plt.plot(val_loss, label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')